# Hierarchical vs Flat Chunking: A Comparison

This notebook demonstrates the difference between flat (recursive) and hierarchical 
chunking in FFAI's RAG pipeline. We index the same documents with both strategies and 
compare search results, showing how hierarchical chunking provides parent-context enrichment.

**Prerequisites:**
- `pip install "ffai[rag]"`
- A `MISTRAL_API_KEY` environment variable

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from ffai.rag import RAG  # noqa: E402
from ffai.rag.embed import Embeddings  # noqa: E402
from ffai.rag.splitters.hierarchical import HierarchicalChunker  # noqa: E402
from ffai.rag.store import VectorStore  # noqa: E402

print(f"Project root: {_project_root}")
print("Imports successful")

## Sample Documents

We use three documents covering different topics to make search results unambiguous.

In [ ]:
DOCUMENTS = {
    "python": (
        "Python is a high-level programming language known for its readability and versatility. "
        "It supports multiple programming paradigms including procedural, object-oriented, and "
        "functional programming. Python has a large standard library and a thriving ecosystem of "
        "third-party packages. The language emphasizes code readability with its notable use of "
        "significant indentation. Python's dynamic typing and interpreted nature make it ideal "
        "for rapid prototyping and scripting."
    ),
    "rust": (
        "Rust is a systems programming language focused on safety, speed, and concurrency. "
        "It enforces memory safety without a garbage collector using a borrow checker system. "
        "Rust has been adopted by major companies for performance-critical applications. "
        "The language provides zero-cost abstractions, move semantics, and guaranteed memory "
        "safety. Rust's type system and ownership model prevent null pointer dereferences, "
        "buffer overflows, and data races at compile time."
    ),
    "cooking": (
        "Italian cuisine is known for its regional diversity and simplicity of preparation. "
        "Popular dishes include pasta, pizza, risotto, and gelato. Italian cooking emphasizes "
        "fresh ingredients and traditional techniques passed down through generations. "
        "The Mediterranean diet, rooted in Italian cooking, is associated with numerous health "
        "benefits including reduced risk of heart disease and improved longevity."
    ),
}

for name, text in DOCUMENTS.items():
    print(f"{name}: {len(text)} chars, {len(text.split())} words")

## Setup: Two RAG Instances

We create two identical RAG pipelines -- one with recursive (flat) chunking, one with hierarchical. Both use the same embedding model and store documents in separate ChromaDB collections.

In [ ]:
import tempfile

api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    print("WARNING: MISTRAL_API_KEY not set. Skipping live API calls.")
    print("The rest of this notebook will show chunking behavior without embeddings.")

tmp_dir = tempfile.mkdtemp()
embed = Embeddings("mistral/mistral-embed", api_key=api_key, cache_enabled=True)

store_flat = VectorStore(collection_name="flat_comparison", dir=f"{tmp_dir}/flat_db")
store_hier = VectorStore(collection_name="hier_comparison", dir=f"{tmp_dir}/hier_db")

rag_flat = RAG(embed=embed, store=store_flat, chunker="recursive", chunk_size=200, chunk_overlap=50)
rag_hier = RAG(embed=embed, store=store_hier, chunker="hierarchical", chunk_size=200, chunk_overlap=50)

print(f"Flat chunker: {rag_flat._chunker_name}")
print(f"Hier chunker: {rag_hier._chunker_name}")

## Step 1: Compare Chunk Counts

Before indexing into the vector store, let's see how each chunker splits the same document.

In [ ]:
text = DOCUMENTS["python"] + "\n\n" + DOCUMENTS["rust"]

from ffai.rag.splitters.recursive import RecursiveChunker  # noqa: E402

recursive = RecursiveChunker(chunk_size=200, chunk_overlap=50)
hierarchical = HierarchicalChunker(chunk_size=200, chunk_overlap=50, parent_chunk_size=400)

flat_chunks = recursive.chunk(text)
hier_chunks = hierarchical.chunk(text)

hier_parents = [c for c in hier_chunks if c.hierarchy_level == 0]
hier_children = [c for c in hier_chunks if c.hierarchy_level > 0]

print(f"Recursive chunks (flat):    {len(flat_chunks)}")
print(f"Hierarchical total chunks:  {len(hier_chunks)} ({len(hier_parents)} parents + {len(hier_children)} children)")
print()
print("Flat chunks:")
for i, c in enumerate(flat_chunks):
    print(f"  [{i}] {len(c.content):3d} chars: {c.content[:60]}...")
print()
print("Hierarchical chunks (parents):")
for c in hier_parents:
    print(f"  P {c.id[:8]}: {len(c.content):3d} chars: {c.content[:60]}...")
print()
print("Hierarchical chunks (children, indexed for search):")
for c in hier_children:
    print(f"  C {c.id[:8]} (parent={c.parent_id[:8]}): {len(c.content):3d} chars: {c.content[:60]}...")

## Step 2: Index Documents

Index all three documents into both pipelines. The hierarchical pipeline stores only child chunks.

In [ ]:
if api_key:
    for name, text in DOCUMENTS.items():
        flat_count = rag_flat.index(text, source=name)
        hier_count = rag_hier.index(text, source=name)
        print(f"{name:10s}: flat={flat_count} chunks, hierarchical={hier_count} children indexed")

    print(f"\nTotal in flat store:  {rag_flat.count()}")
    print(f"Total in hier store:  {rag_hier.count()}")
else:
    print("Skipping indexing (no API key). Set MISTRAL_API_KEY to run live.")

## Step 3: Compare Search Results

Search both pipelines for the same query and compare the results. The key difference: hierarchical results include `parent_content`.

In [ ]:
if api_key:
    query = "memory safety borrow checker"

    flat_hits = rag_flat.search(query, top_k=3)
    hier_hits = rag_hier.search(query, top_k=3)

    print(f"Query: '{query}'")
    print(f"\n{'='*60}")
    print(f"FLAT (recursive) -- {len(flat_hits)} hits")
    print(f"{'='*60}")
    for i, h in enumerate(flat_hits):
        parent_info = f"\n  parent_content: {h.parent_content}" if h.parent_content else "\n  parent_content: None"
        print(f"  [{i}] source={h.source}, score={h.score:.3f}")
        print(f"  content: {h.content[:80]}...{parent_info}")

    print(f"\n{'='*60}")
    print(f"HIERARCHICAL -- {len(hier_hits)} hits")
    print(f"{'='*60}")
    for i, h in enumerate(hier_hits):
        parent_info = f"\n  parent_content: {h.parent_content[:80]}..." if h.parent_content else "\n  parent_content: None"
        print(f"  [{i}] source={h.source}, score={h.score:.3f}")
        print(f"  content: {h.content[:80]}...{parent_info}")
else:
    print("Skipping search (no API key). The chunking comparison above works without embeddings.")

## Step 4: Query with Context

Use `format_hits()` to see how the hierarchical results include parent context in the formatted output sent to the LLM.

In [ ]:
if api_key:
    from ffai.rag.format import format_hits

    flat_context = format_hits(flat_hits[:2])
    hier_context = format_hits(hier_hits[:2])

    print("FLAT formatted context:")
    print("-" * 40)
    print(flat_context[:400])
    print("...")
    print()
    print("HIERARCHICAL formatted context:")
    print("-" * 40)
    print(hier_context[:600])
    print("...")
else:
    print("Skipping context formatting (no API key).")

## Summary

| Aspect | Flat (recursive) | Hierarchical |
|--------|------------------|-------------|
| Chunks indexed | All chunks | Child chunks only |
| Embeddings computed | One per chunk | One per child chunk |
| Search granularity | Chunk-level | Child-level |
| Context enrichment | None | Parent content included |
| Best for | Short documents, uniform content | Long documents with sections |

**When to use hierarchical:**
- Long documents (contracts, papers, manuals) where a matched paragraph needs surrounding section context
- When you want the LLM to see both the specific match and the broader context
- When storage efficiency matters (no redundant parent embeddings)

**When to stick with recursive:**
- Short documents where each chunk is self-contained
- When you don't need parent-child context relationships